# 06 · Platform Agent

Pablo's orchestrator and rule keeper:

- **CLARIFY**: a code rule decides whether to ask; the LLM only words the question. At most
  one question per request.
- **Routers** for the three bounded loops: INTENT → CLARIFY, EVALUATING → SEARCHING (one
  retry), RESPONDING → RESPONDING (one Humanizer retry).
- The policy rules themselves live in `src/tools/policy.py` (notebook 01).

Changes from Pablo's version:

- **Printing question.** The plan asks for a question when several printings match
  ("Charizard" could be Base Set or 151). The old rule for this could never run; now
  `needs_clarification` takes the printings found by the Merchant and asks which one, by
  set name. Budget and printing can be asked together in one question.
- Shared LLM client, with a fixed template if the LLM is unavailable (and offline).

In [ ]:
#| default_exp agents.platform

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Platform Agent (Orchestrator): CLARIFY logic, policy gates, and bounded-loop routing."""

import json
import time
from collections.abc import Callable
from typing import Any, Literal

from src.config import load_config
from src.llm import LLM, LLMError, make_llm, offline_handler
from src.pipeline import between
from src.schemas import ParsedQuery
from src.state import AgentState

## CLARIFY

In [ ]:
#| export
PrintingsFn = Callable[[ParsedQuery], list[str]]

In [ ]:
#| export
def needs_clarification(
    parsed: ParsedQuery | None,
    clarify_count: int,
    max_clarify: int = 1,
    printings: list[str] | None = None,
) -> str | None:
    """Deterministic code rule deciding whether to enter CLARIFY (Section 3): the card is
    missing, the budget is missing (unless the question is consultative), or several
    printings match and the buyer named no set.

    Returns a reason string if clarification is needed, or None to skip directly to SEARCHING.
    """
    if clarify_count >= max_clarify:
        return None
    if parsed is None or not parsed.card_name:
        return (
            "missing_card_and_budget" if parsed is None or parsed.budget is None else "missing_card"
        )
    no_budget = parsed.budget is None and not parsed.is_consultative
    ambiguous = (
        printings is not None
        and len(printings) > 1
        and not parsed.set_code
        and not parsed.any_printing
    )
    if ambiguous and no_budget:
        return "ambiguous_printing_and_budget"
    if ambiguous:
        return "ambiguous_printing"
    if no_budget:
        return "missing_budget"
    return None

In [ ]:
#| export
def clarify_template(reason: str, card_name: str | None, printings: list[str]) -> str:
    """Fixed wording, used offline and whenever the LLM is unavailable."""
    options = " or ".join(p.split(" from ", 1)[-1] for p in printings[:3])
    card = card_name or "card"
    if reason == "ambiguous_printing_and_budget":
        return f"Which {card} do you mean, {options}? And what's your budget?"
    if reason == "ambiguous_printing":
        return f"Which {card} do you mean, {options}?"
    if reason == "missing_budget":
        return f"What's the most you'd like to spend on the {card}?"
    if reason == "missing_card":
        return "Which card are you looking for?"
    return "Which card are you looking for, and what's your budget?"

In [ ]:
#| export
CLARIFY_SYSTEM = """You are a helpful Pokemon TCG shop assistant. Ask ONE short, natural
clarifying question (under 25 words) to resolve what is missing from the buyer's request.
If printings are listed, offer them by set name as the choices. Ask nothing else."""

In [ ]:
#| export
def _phrase_clarifying_question_llm(
    llm: LLM, user_input: str, reason: str, card_name: str | None, printings: list[str]
) -> str:
    """Use the LLM to phrase one concise, spoken-friendly clarifying question."""
    facts = {"request": user_input, "missing": reason, "card": card_name, "printings": printings}
    prompt = "Facts (JSON):\n<facts>\n" + json.dumps(facts) + "\n</facts>\nQuestion:"
    return llm.text(tag="clarify", system=CLARIFY_SYSTEM, prompt=prompt)

In [ ]:
#| export
def make_clarify_node(
    config: dict[str, Any] | None = None,
    question_phrasing_fn: Callable[[str, str], str] | None = None,
    llm: LLM | None = None,
    printings_fn: PrintingsFn | None = None,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the CLARIFY node (Platform Agent: code rule + LLM wording)."""
    cfg = config or load_config()
    max_clarify = int(cfg["limits"]["max_clarify_questions"])
    model = llm or (None if question_phrasing_fn else make_llm(cfg))

    def clarify_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        parsed = state.get("parsed_query")
        clarify_count = int(state.get("clarify_count", 0))
        printings = printings_fn(parsed) if (printings_fn and parsed and parsed.card_name) else []
        reason = (
            needs_clarification(parsed, clarify_count, max_clarify, printings) or "missing_details"
        )
        card = parsed.card_name if parsed else None

        if question_phrasing_fn is not None:
            question = question_phrasing_fn(state.get("user_input", ""), reason)
        else:
            try:
                question = _phrase_clarifying_question_llm(
                    model, state.get("user_input", ""), reason, card, printings
                )
            except LLMError:
                question = clarify_template(reason, card, printings)

        timings = dict(state.get("stage_timings", {}))
        timings["CLARIFY"] = round(time.perf_counter() - t0, 4)

        return {
            "current_state": "CLARIFY",
            "clarify_count": clarify_count + 1,
            "clarifying_question": question,
            "stage_timings": timings,
        }

    return clarify_node

## Routers (LangGraph conditional edges)

In [ ]:
#| export
def route_after_intent(
    state: AgentState,
    max_clarify: int = 1,
    printings_fn: PrintingsFn | None = None,
) -> Literal["CLARIFY", "SEARCHING"]:
    """Route to CLARIFY if request is unclear and clarify_count < max_clarify; else SEARCHING."""
    parsed = state.get("parsed_query")
    printings = printings_fn(parsed) if (printings_fn and parsed and parsed.card_name) else None
    reason = needs_clarification(
        parsed,
        clarify_count=int(state.get("clarify_count", 0)),
        max_clarify=max_clarify,
        printings=printings,
    )
    return "CLARIFY" if reason is not None else "SEARCHING"

In [ ]:
#| export
def route_after_evaluating(
    state: AgentState,
    risk_reject_threshold: float = 0.6,
    max_search_retries: int = 1,
) -> Literal["SEARCHING", "RANKING"]:
    """Only EVALUATING can send work backward to SEARCHING, and at most once (Section 3)."""
    evaluated = state.get("evaluated_listings", [])
    surviving = [item for item in evaluated if item.risk_score < risk_reject_threshold]
    retries = int(state.get("search_retry_count", 0))

    if not surviving and retries < max_search_retries:
        return "SEARCHING"
    return "RANKING"

In [ ]:
#| export
def route_after_responding(
    state: AgentState,
    max_humanizer_retries: int = 1,
) -> Literal["RESPONDING", "DONE"]:
    """If Humanizer number check fails, retry once; otherwise proceed to DONE.

    The Humanizer increments humanizer_retry_count each time its check fails, so with the
    default of 1 the LLM gets two attempts before the template takes over.
    """
    passed = bool(state.get("number_check_passed", False))
    retries = int(state.get("humanizer_retry_count", 0))
    if not passed and retries <= max_humanizer_retries:
        return "RESPONDING"
    return "DONE"

## Offline stand-in

In [ ]:
#| export
@offline_handler("clarify")
def offline_clarify(_system: str, prompt: str) -> str:
    """Offline mode asks the template question."""
    f = json.loads(between(prompt, "facts"))
    return clarify_template(f["missing"], f.get("card"), f.get("printings") or [])

# Try it

In [ ]:
from src.agents.merchant import distinct_printings
from src.config import load_config
from src.llm import make_llm
from src.pipeline import load_data

cfg = load_config()
data = load_data(cfg)
llm = make_llm(cfg)


def printings(q):
    return distinct_printings(data.listings, q.card_name, cfg)


for q in [
    ParsedQuery(card_name="Charizard", budget=100),
    ParsedQuery(card_name="Charizard"),
    ParsedQuery(card_name="Mewtwo"),
    ParsedQuery(card_name="Mewtwo", budget=50),
]:
    print(q.card_name, q.budget, "->", needs_clarification(q, 0, 1, printings(q)))

In [ ]:
node = make_clarify_node(cfg, llm=llm, printings_fn=printings)
node({"parsed_query": ParsedQuery(card_name="Charizard"), "user_input": "I want a Charizard"})[
    "clarifying_question"
]

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()